# QDM Bias Correction — CanESM5 vs ERA5

Trains Quantile Delta Mapping (Cannon et al. 2015) on ERA5 1980–2010 vs CanESM5 historical,  
then applies it to SSP245 and SSP585 (2015–2100).

| Variable | Transform | Unit |
|----------|-----------|------|
| tas      | additive  | K    |
| tasmax   | additive  | K    |
| sfcWind  | log + additive | m s⁻¹ |
| rsds     | log + additive | W m⁻² |

In [ ]:
import glob
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from tqdm.notebook import tqdm
from xclim import sdba
from xclim.sdba import processing as sdba_proc
from xarray.coding.calendar_ops import convert_calendar

ERA5_DIR   = Path("../data/raw/era5_daily")
CMIP_DIR   = Path("../data/raw/CanESM5")
OUTPUT_DIR = Path("../data/proc/cmip6_bc")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GCM   = "CanESM5"
RUN   = "r10i1p1f1"
SSPS  = ["ssp245", "ssp585"]
TRAIN = slice("1980-01-01", "2010-12-31")

## 1. Load ERA5 reference (per-variable files)

In [ ]:
def load_era5(vname, units):
    ds = xr.open_dataset(ERA5_DIR / f"era5_{vname}_1980_2020.nc")
    da = ds[vname]
    da.attrs["units"] = units
    return da.sortby("lat").sortby("lon")

era5 = {
    "tas":     load_era5("tas",     "K"),
    "tasmax":  load_era5("tasmax",  "K"),
    "sfcWind": load_era5("sfcWind", "m s-1"),
    "rsds":    load_era5("rsds",    "W m-2"),
}

# Convert Gregorian → noleap to match CMIP6
era5 = {v: convert_calendar(da, "noleap", align_on="date").sel(time=TRAIN)
        for v, da in era5.items()}

print("ERA5 loaded:")
for v, da in era5.items():
    print(f"  {v}: {da.sizes}  {da.time.values[0].astype('datetime64[D]')} → {da.time.values[-1].astype('datetime64[D]')}")

## 2. Load CanESM5 historical

In [ ]:
def load_cmip(var, scenario, units):
    pattern = str(CMIP_DIR / f"{var}_day_{GCM}_{scenario}_{RUN}*_india.nc")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files: {pattern}")
    da = xr.open_mfdataset(files, combine="by_coords")[var]
    da = da.drop_vars([c for c in da.coords
                       if c not in {"time","lat","lon","latitude","longitude"}],
                      errors="ignore")
    da["time"] = da.indexes["time"].floor("D")
    da.attrs["units"] = units
    return da.sortby("lat").sortby("lon").sortby("time")

hist_uas = load_cmip("uas", "historical", "m s-1")
hist_vas = load_cmip("vas", "historical", "m s-1")
hist_wind = np.hypot(hist_uas, hist_vas).rename("sfcWind")
hist_wind.attrs["units"] = "m s-1"

hist = {
    "tas":     load_cmip("tas",    "historical", "K"),
    "tasmax":  load_cmip("tasmax", "historical", "K"),
    "sfcWind": hist_wind,
    "rsds":    load_cmip("rsds",   "historical", "W m-2"),
}
hist = {v: da.sel(time=TRAIN) for v, da in hist.items()}

print("Historical loaded:")
for v, da in hist.items():
    print(f"  {v}: {da.sizes}")

## 3. Align grids and time axes

In [ ]:
# Regrid ERA5 (0.25°) → CanESM5 grid (~2.8°)
ref_lat = hist["tas"].lat
ref_lon = hist["tas"].lon

era5_rg = {v: da.interp(lat=ref_lat, lon=ref_lon, method="linear")
           for v, da in era5.items()}

# Align time: find common dates between ERA5 (noleap) and CMIP6
era5_dates  = pd.DatetimeIndex(era5_rg["tas"].time.values).normalize()
hist_dates  = pd.DatetimeIndex([pd.Timestamp(str(t)[:10]) for t in hist["tas"].time.values])
common      = era5_dates.intersection(hist_dates)
print(f"Common training days: {len(common)}")

era5_rg = {v: da.sel(time=[t for t in da.time.values
                            if pd.Timestamp(str(t)[:10]) in common])
           for v, da in era5_rg.items()}
hist    = {v: da.sel(time=[t for t in da.time.values
                            if pd.Timestamp(str(t)[:10]) in common])
           for v, da in hist.items()}

# Match time coordinate so xclim can align
for v in era5_rg:
    era5_rg[v] = era5_rg[v].assign_coords(time=hist[v].time.values)

## 4. QDM bias correction — SSP245 and SSP585

In [ ]:
VAR_CFG = {
    # var    kind   log    jitter  units
    "tas":     ("+", False, None,   "K"),
    "tasmax":  ("+", False, None,   "K"),
    "sfcWind": ("+", True,  1e-6,   "m s-1"),
    "rsds":    ("+", True,  1e-6,   "W m-2"),
}

def jitter_log(da, lower, unit):
    da = sdba_proc.jitter(da, lower=f"{lower} {unit}", minimum=f"0 {unit}")
    return sdba_proc.to_additive_space(da, lower_bound=f"0 {unit}", trans="log")

outer = tqdm(SSPS, desc="SSP", position=0)
for ssp in outer:
    outer.set_postfix(ssp=ssp)

    # Load future SSP
    fut_uas  = load_cmip("uas", ssp, "m s-1")
    fut_vas  = load_cmip("vas", ssp, "m s-1")
    fut_wind = np.hypot(fut_uas, fut_vas).rename("sfcWind")
    fut_wind.attrs["units"] = "m s-1"
    fut = {
        "tas":     load_cmip("tas",    ssp, "K"),
        "tasmax":  load_cmip("tasmax", ssp, "K"),
        "sfcWind": fut_wind,
        "rsds":    load_cmip("rsds",   ssp, "W m-2"),
    }

    inner = tqdm(VAR_CFG.items(), desc="  var", position=1, leave=False)
    for vname, (kind, use_log, jitter_low, unit) in inner:
        inner.set_postfix(var=vname)
        out_path = OUTPUT_DIR / f"{vname}_{GCM}_{ssp}_bc.nc"
        if out_path.exists():
            print(f"  {ssp}/{vname}: exists — skipping")
            continue

        ref_v  = era5_rg[vname].load()
        hist_v = hist[vname].load()
        fut_v  = fut[vname].load()

        if use_log:
            ref_v  = jitter_log(ref_v,  jitter_low, unit)
            hist_v = jitter_log(hist_v, jitter_low, unit)
            fut_v  = jitter_log(fut_v,  jitter_low, unit)

        QM = sdba.QuantileDeltaMapping.train(
            ref_v, hist_v,
            nquantiles=50,
            kind=kind,
            group="time.month",
        )
        bc_v = QM.adjust(fut_v, interp="linear", extrapolation="constant")

        if use_log:
            bc_v = sdba_proc.from_additive_space(bc_v)
            bc_v.values[bc_v.values < 1e-5] = 0.0

        ds_out = bc_v.to_dataset(name=vname)
        ds_out[vname].attrs["units"] = unit
        ds_out.attrs = {
            "description": f"QDM bias-corrected {vname} — {GCM} {RUN} {ssp}",
            "method": "Quantile Delta Mapping (Cannon et al. 2015) via xclim.sdba",
            "reference": "ERA5 daily 1980–2010",
            "gcm": GCM, "run": RUN, "ssp": ssp,
        }
        ds_out.to_netcdf(out_path)
        print(f"  → {out_path.name}")

print("\nBias correction complete.")

## 5. Extract Global Warming Level (GWL) windows

In [ ]:
wl_dir = Path("../aux_data/mathause-cmip_warming_levels-f47853e/warming_levels/cmip6")
wl_files = (sorted(glob.glob(str(wl_dir / "csv" / "*1850_1900*.csv"))) or
            sorted(glob.glob(str(wl_dir / "**" / "*1850_1900*.csv"), recursive=True)))
wl_files.sort(key=lambda f: "all" not in f)

if not wl_files:
    print("Warming levels CSV not found — skipping")
else:
    wl = pd.read_csv(wl_files[0], comment="#", skipinitialspace=True)
    wl.columns = wl.columns.str.strip()
    print(f"Loaded: {wl_files[0]}\nColumns: {list(wl.columns)}")

    gwl_dir = OUTPUT_DIR / "gwl"
    gwl_dir.mkdir(exist_ok=True)

    rows = []
    for ssp in tqdm(SSPS, desc="SSP"):
        for gwl in [1.5, 2.0, 3.0, 4.0]:
            mask = ((wl["model"].str.strip() == GCM) &
                    (wl["exp"].str.strip() == ssp) &
                    (wl["warming_level"].astype(float) == gwl))
            if "ensemble" in wl.columns:
                mask &= wl["ensemble"].str.strip() == RUN
            row = wl[mask]
            if row.empty:
                row = wl[(wl["model"].str.strip() == GCM) &
                         (wl["exp"].str.strip() == ssp) &
                         (wl["warming_level"].astype(float) == gwl)]
            if row.empty:
                continue
            start, end = int(row.iloc[0]["start_year"]), int(row.iloc[0]["end_year"])
            rows.append({"ssp": ssp, "gwl": gwl, "start": start, "end": end})

            for vname in VAR_CFG:
                bc_file = OUTPUT_DIR / f"{vname}_{GCM}_{ssp}_bc.nc"
                out_gwl = gwl_dir / f"{vname}_{GCM}_{ssp}_GWL{str(gwl).replace('.','')}.nc"
                if not bc_file.exists() or out_gwl.exists():
                    continue
                ds = xr.open_dataset(bc_file)
                ds.sel(time=slice(str(start), str(end))).to_netcdf(out_gwl)

    pd.DataFrame(rows).to_csv(OUTPUT_DIR / "gwl_windows.csv", index=False)
    print(pd.DataFrame(rows).to_string(index=False))

## 6. Quick validation — domain-mean time series

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

PLOT_VARS = {
    "tas":     ("tas (K)",       "K"),
    "tasmax":  ("tasmax (K)",    "K"),
    "sfcWind": ("sfcWind (m/s)", "m s-1"),
    "rsds":    ("rsds (W/m²)",   "W m-2"),
}
COLORS = {"ssp245": "steelblue", "ssp585": "firebrick"}

for ax, (vname, (label, _)) in zip(axes, PLOT_VARS.items()):
    # ERA5 reference (annual mean, domain average)
    era5_da = xr.open_dataset(ERA5_DIR / f"era5_{vname}_1980_2020.nc")[vname]
    era5_ann = era5_da.mean(["lat", "lon"]).resample(time="YE").mean()
    ax.plot(era5_ann.time.dt.year, era5_ann.values, "k-", lw=1.5, label="ERA5", zorder=3)

    for ssp in SSPS:
        bc_file = OUTPUT_DIR / f"{vname}_{GCM}_{ssp}_bc.nc"
        if not bc_file.exists():
            continue
        bc = xr.open_dataset(bc_file)[vname]
        bc_ann = bc.mean(["lat", "lon"]).resample(time="YE").mean()
        ax.plot(bc_ann.time.dt.year, bc_ann.values,
                color=COLORS[ssp], lw=1, alpha=0.8, label=ssp)

    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.set_xlabel("Year")

plt.suptitle(f"Domain-mean annual series — {GCM} {RUN}", fontsize=12)
plt.tight_layout()
plt.show()